In [1]:
"""usage demo for qiskit_ibm_transpiler Pauli networkx synthesis via RL, https://quantum.cloud.ibm.com/docs/en/guides/ai-transpiler-passes"""

from qiskit.circuit.library import QAOAAnsatz
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import PassManager
from qiskit.circuit.library import EfficientSU2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.collection import CollectPauliNetworks
from qiskit_ibm_transpiler.ai.synthesis import AIPauliNetworkSynthesis
from qiskit_ibm_runtime import QiskitRuntimeService

TOKEN = "arS6k_KG_M4QxzK6oDnAajNNBrneH8XTVJZXiMYSv4lk"  # ! My API

ibm_torino = QiskitRuntimeService(channel="ibm_cloud", token=TOKEN).backend("ibm_torino")

import sys

sys.path.append("..")


# A simple Hamiltonian for demonstration: H = Z0Z1 + Z1Z2 + Z2Z0
hamiltonian = SparsePauliOp.from_list([("ZZI", 1), ("IZZ", 1), ("ZIZ", 1)])
ansatz = QAOAAnsatz(cost_operator=hamiltonian, reps=2)
circuit = ansatz.decompose()

qiskit_runtime_service._discover_account:WARNING:2025-11-19 18:24:37,244: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2025-11-19 18:24:47,402: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2025-11-19 18:24:47,402: Using instance: open-instance, plan: open


In [2]:
circuit = circuit.decompose()
circuit.draw(fold=-1)

┌─────────┐                                             ┌──────────────┐                                             ┌──────────────┐
q_0: ┤ U2(0,π) ├────────────────■──────────────■─────────────┤ Rx(2.0*β[0]) ├────────────────■──────────────■─────────────┤ Rx(2.0*β[1]) ├
     ├─────────┤                │ZZ(2.0*γ[0])  │             ├──────────────┤                │ZZ(2.0*γ[1])  │             ├──────────────┤
q_1: ┤ U2(0,π) ├─■──────────────■──────────────┼─────────────┤ Rx(2.0*β[0]) ├─■──────────────■──────────────┼─────────────┤ Rx(2.0*β[1]) ├
     ├─────────┤ │ZZ(2.0*γ[0])                 │ZZ(2.0*γ[0]) ├──────────────┤ │ZZ(2.0*γ[1])                 │ZZ(2.0*γ[1]) ├──────────────┤
q_2: ┤ U2(0,π) ├─■─────────────────────────────■─────────────┤ Rx(2.0*β[0]) ├─■─────────────────────────────■─────────────┤ Rx(2.0*β[1]) ├
     └─────────┘                                             └──────────────┘                                             └──────────────┘

In [3]:
# circuit = EfficientSU2(10, entanglement="full", reps=1).decompose()

# Build the AI transpilation pipeline
ai_passmanager = PassManager(
    [
        # First, route the circuit to the target backend topology
        AIRouting(backend=ibm_torino, optimization_level=3, layout_mode="optimize"),
        # Collect Pauli Network blocks (H, S, SX, CX, RX, RY, RZ gates)
        # Supports up to 6-qubit blocks
        CollectPauliNetworks(do_commutative_analysis=True, min_block_size=4, max_block_size=6, num_reps=10),
        # Apply RL-based synthesis to optimize the collected blocks
        # Uses reinforcement learning models trained to minimize gate count and depth
        AIPauliNetworkSynthesis(
            backend=ibm_torino,  # Target backend for connectivity constraints
            replace_only_if_better=True,  # Only replace if RL synthesis improves the circuit
            max_threads=10,  # Parallel synthesis requests
        ),
    ]
)

In [7]:
circuit = circuit.decompose()

circuit.draw(fold=-1)

┌─────────────┐                                                                               ┌───────────────┐                                                                               ┌───────────────┐
q_0: ┤ U3(π/2,0,π) ├────────────────────────────■────────────────────■────■─────────────────────■──┤ R(2.0*β[0],0) ├────────────────────────────■────────────────────■────■─────────────────────■──┤ R(2.0*β[1],0) ├
     ├─────────────┤                          ┌─┴─┐┌──────────────┐┌─┴─┐  │  ┌───────────────┐  │  └───────────────┘                          ┌─┴─┐┌──────────────┐┌─┴─┐  │  ┌───────────────┐  │  └───────────────┘
q_1: ┤ U3(π/2,0,π) ├──■────────────────────■──┤ X ├┤ Rz(2.0*γ[0]) ├┤ X ├──┼──┤ R(2.0*β[0],0) ├──┼─────────────────────■────────────────────■──┤ X ├┤ Rz(2.0*γ[1]) ├┤ X ├──┼──┤ R(2.0*β[1],0) ├──┼───────────────────
     ├─────────────┤┌─┴─┐┌──────────────┐┌─┴─┐└───┘└──────────────┘└───┘┌─┴─┐└┬──────────────┤┌─┴─┐┌───────────────┐┌─┴─┐┌──────────────┐┌─┴─┐└───┘└──────────────┘└───┘┌─┴─┐└┬──────────────┤┌─┴─┐┌───────────────┐
q_2: ┤ U3(π/2,0,π) ├┤ X ├┤ Rz(2.0*γ[0]) ├┤ X ├──────────────────────────┤ X ├─┤ Rz(2.0*γ[0]) ├┤ X ├┤ R(2.0*β[0],0) ├┤ X ├┤ Rz(2.0*γ[1]) ├┤ X ├──────────────────────────┤ X ├─┤ Rz(2.0*γ[1]) ├┤ X ├┤ R(2.0*β[1],0) ├
     └─────────────┘└───┘└──────────────┘└───┘                          └───┘ └──────────────┘└───┘└───────────────┘└───┘└──────────────┘└───┘                          └───┘ └──────────────┘└───┘└───────────────┘

In [8]:
transpiled_circuit = ai_passmanager.run(circuit)

print(f"Original circuit - Gates: {circuit.num_nonlocal_gates()}, Depth: {circuit.depth()}")
print(f"Optimized circuit - Gates: {transpiled_circuit.num_nonlocal_gates()}, Depth: {transpiled_circuit.depth()}")

INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running Pauli Network AI synthesis on local mode
INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running Pauli Network AI synthesis on local mode
INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running Pauli Network AI synthesis on local mode


Original circuit - Gates: 12, Depth: 21
Optimized circuit - Gates: 14, Depth: 24


In [6]:
transpiled_circuit.draw(fold=-1)

Qubit(QuantumRegister(133, 'q'), 26) -> 0 ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                                                                                                                                                                                         
   Qubit(QuantumRegister(133, 'q'), 27) -> 1 ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                                                                                                                                                                                         
   Qubit(QuantumRegister(133, 'q'), 28) -> 2 ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                                                                                                                                                                                         
    Qubit(QuantumRegister(133, 'q'), 0) -> 3 ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                                                                                                                                                                                         
  Qubit(QuantumRegister(133, 'q'), 125) -> 4 ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                                                                                                                                                                                         
   Qubit(QuantumRegister(133, 'q'), 45) -> 5 ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                                                                                                                                                                                         
  Qubit(QuantumRegister(133, 'q'), 127) -> 6 ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                                                                                                                                                                                         
   Qubit(QuantumRegister(133, 'q'), 51) -> 7 ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                                                                                                                                                                                         
   Qubit(QuantumRegister(133, 'q'), 84) -> 8 ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                                                                                                                                                                                         
   Qubit(QuantumRegister(133, 'q'), 43) -> 9 ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                                                                                                                                                     

In [20]:
transpiled_circuit = ai_passmanager.run(circuit)

print(f"Original circuit - Gates: {circuit.num_nonlocal_gates()}, Depth: {circuit.depth()}")
print(f"Optimized circuit - Gates: {transpiled_circuit.num_nonlocal_gates()}, Depth: {transpiled_circuit.depth()}")

INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running Pauli Network AI synthesis on local mode
INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running Pauli Network AI synthesis on local mode
INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running Pauli Network AI synthesis on local mode
INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running Pauli Network AI synthesis on local mode
INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running Pauli Network AI synthesis on local mode
INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running Pauli Network AI synthesis on local mode
INFO:qiskit_ibm_transpiler.wrappers.ai_local_synthesis:Running Pauli Network AI synthesis on local mode


Original circuit - Gates: 45, Depth: 21
Optimized circuit - Gates: 85, Depth: 49


In [22]:
circuit.draw(fold=-1)

┌──────────┐┌───────────┐                                                                                                                             ┌───────────┐┌───────────┐                                                                                                                                                                                                                    
q_0: ┤ Ry(θ[0]) ├┤ Rz(θ[10]) ├──■────■─────────■─────────■──────────────■──────────────■───────────────────■───────────────────■────────────────────────■──┤ Ry(θ[20]) ├┤ Rz(θ[30]) ├────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     ├──────────┤├───────────┤┌─┴─┐  │         │         │              │              │                   │                   │                        │  └───────────┘└───────────┘          ┌───────────┐┌───────────┐                                                                                                                                                                                
q_1: ┤ Ry(θ[1]) ├┤ Rz(θ[11]) ├┤ X ├──┼────■────┼────■────┼─────────■────┼─────────■────┼──────────────■────┼──────────────■────┼───────────────────■────┼───────────────────────────────────■──┤ Ry(θ[21]) ├┤ Rz(θ[31]) ├────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     ├──────────┤├───────────┤└───┘┌─┴─┐┌─┴─┐  │    │    │         │    │         │    │              │    │              │    │                   │    │                                   │  └───────────┘└───────────┘          ┌───────────┐┌───────────┐                                                                                                                                            
q_2: ┤ Ry(θ[2]) ├┤ Rz(θ[12]) ├─────┤ X ├┤ X ├──┼────┼────┼────■────┼────┼────■────┼────┼─────────■────┼────┼─────────■────┼────┼──────────────■────┼────┼──────────────────────────────■────┼───────────────────────────────────■──┤ Ry(θ[22]) ├┤ Rz(θ[32]) ├────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     ├──────────┤├───────────┤     └───┘└───┘┌─┴─┐┌─┴─┐  │  ┌─┴─┐  │    │    │    │    │         │    │    │         │    │    │              │    │    │                              │    │                                   │  └───────────┘└───────────┘     ┌───────────┐┌───────────┐                                                                                                             
q_3: ┤ Ry(θ[3]) ├┤ Rz(θ[13]) ├───────────────┤ X ├┤ X ├──┼──┤ X ├──┼────┼────┼────┼────┼────■────┼────┼────┼────■────┼────┼────┼─────────■────┼────┼────┼─────────────────────■────────┼────┼──────────────────────────────■────┼──────────────────────────────■──┤ Ry(θ[23]) ├┤ Rz(θ[33]) ├─────────────────────────────────────────────────────────────────────────────────────────────────────────────
     ├──────────┤├───────────┤               └───┘└───┘┌─┴─┐└───┘┌─┴─┐  │  ┌─┴─┐  │    │  ┌─┴─┐  │    │    │    │    │    │    │         │    │    │    │                     │        │    │                              │    │                              │  └───────────┘└───────────┘     ┌───────────┐┌───────────┐                                                                              
q_4: ┤ Ry(θ[4]) ├┤ Rz(θ[14]) ├─────────────────────────┤ X ├─────┤ X ├──┼──┤ X ├──┼────┼──┤ X ├──┼────┼────┼────┼────┼────┼────┼────■────┼────┼────┼────┼────────■────────────┼────────┼────┼─────────────────────■────────┼────┼─────────────────────■────────┼──────────────────────────────■──┤ Ry(θ[24]) ├┤ Rz(θ[34]) ├──────────────────────────────────────────────────────────────────────────────
     ├──────────┤├───────────┤                         └───┘     

In [23]:
import qiskit

transpiled_circuit_classical = qiskit.transpile(
    circuit, optimization_level=3, basis_gates=["u1", "u2", "u3", "cx"], backend=ibm_torino, layout_method="sabre"
)

print(f"Original circuit - Gates: {circuit.num_nonlocal_gates()}, Depth: {circuit.depth()}")
print(
    f"Optimized circuit - Gates: {transpiled_circuit_classical.num_nonlocal_gates()}, Depth: {transpiled_circuit_classical.depth()}"
)

Original circuit - Gates: 45, Depth: 21
Optimized circuit - Gates: 118, Depth: 87


In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import PauliEvolutionGate
import json


with open("../benchmarks/uccsd_json/CH2_frz_BK_sto3g.json", "r") as f:
    data = json.load(f)
paulis = data["paulis"][:]
coeffs = data["coeffs"]

n = len(paulis[0])  # number of qubits
op = SparsePauliOp(paulis, coeffs)
qc = QuantumCircuit(n)
qc.append(PauliEvolutionGate(op), reversed(range(n)))

In [31]:
qc.decompose().draw(fold=-1)

┌────┐                                ┌──────┐┌───┐                                  ┌───┐  ┌────┐                                                          ┌──────┐┌───┐                                                        ┌───┐ ┌────┐                                                ┌──────┐┌───┐                                                        ┌───┐                                                                                                                                                                                                                                                                                                                                ┌────┐                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      ┌──────┐┌────┐                                                                                           ┌──────┐┌────┐                                                                                ┌──────┐┌────┐                                                                                                             ┌──────┐┌───┐                                                                                   ┌───┐┌───┐                                                                                                                ┌───┐  ┌───┐                                                                                                       ┌───┐  ┌───┐                                                                                        ┌───┐┌────┐                                                                                                                 ┌──────┐┌────┐                                                                                           ┌──────┐┌────┐                                                                                          ┌──────┐┌────┐                                                                                                             ┌──────┐┌───┐                                                                                             ┌───┐┌───┐                                                                                                                ┌───┐  ┌───┐                                                                                                                 ┌───┐  ┌───┐                                                                                         ┌───┐┌────┐                                                                                             ┌──────┐┌────┐                                                                                           ┌──────┐┌────┐                                             

In [ ]:
tranpiled_qc = ai_passmanager.run(qc.decompose())

In [ ]:
print(f"Original circuit - Gates: {qc.decompose().num_nonlocal_gates()}, Depth: {qc.decompose().depth()}")
print(f"Optimized circuit - Gates: {tranpiled_qc.num_nonlocal_gates()}, Depth: {tranpiled_qc.depth()}")

In [34]:
tranpiled_qc.draw(fold=-1)

Qubit(QuantumRegister(133, 'q'), 0) -> 0 
                                             
    Qubit(QuantumRegister(133, 'q'), 1) -> 1 
                                             
    Qubit(QuantumRegister(133, 'q'), 2) -> 2 
                                             
    Qubit(QuantumRegister(133, 'q'), 3) -> 3 
                                             
    Qubit(QuantumRegister(133, 'q'), 4) -> 4 
                                             
    Qubit(QuantumRegister(133, 'q'), 5) -> 5 
                                             
    Qubit(QuantumRegister(133, 'q'), 6) -> 6 
                                             
    Qubit(QuantumRegister(133, 'q'), 7) -> 7 
                                             
    Qubit(QuantumRegister(133, 'q'), 8) -> 8 
                                             
    Qubit(QuantumRegister(133, 'q'), 9) -> 9 
                                             
  Qubit(QuantumRegister(133, 'q'), 10) -> 10 
                                             
  Qubit(QuantumRegister(133, 'q'), 11) -> 11 
                                             
  Qubit(QuantumRegister(133, 'q'), 12) -> 12 
                                             
  Qubit(QuantumRegister(133, 'q'), 13) -> 13 
                                             
  Qubit(QuantumRegister(133, 'q'), 14) -> 14 
                                             
  Qubit(QuantumRegister(133, 'q'), 15) -> 15 
                                             
  Qubit(QuantumRegister(133, 'q'), 16) -> 16 
                                             
  Qubit(QuantumRegister(133, 'q'), 17) -> 17 
                                             
  Qubit(QuantumRegister(133, 'q'), 18) -> 18 
                                             
  Qubit(QuantumRegister(133, 'q'), 19) -> 19 
                                             
  Qubit(QuantumRegister(133, 'q'), 20) -> 20 
                                             
  Qubit(QuantumRegister(133, 'q'), 21) -> 21 
                                             
  Qubit(QuantumRegister(133, 'q'), 22) -> 22 
                                             
  Qubit(QuantumRegister(133, 'q'), 23) -> 23 
                                             
  Qubit(QuantumRegister(133, 'q'), 24) -> 24 
                                             
  Qubit(QuantumRegister(133, 'q'), 25) -> 25 
                                             
  Qubit(QuantumRegister(133, 'q'), 26) -> 26 
                                             
  Qubit(QuantumRegister(133, 'q'), 27) -> 27 
                                             
  Qubit(QuantumRegister(133, 'q'), 28) -> 28 
                                             
  Qubit(QuantumRegister(133, 'q'), 29) -> 29 
                                             
  Qubit(QuantumRegister(133, 'q'), 30) -> 30 
                                             
  Qubit(QuantumRegister(133, 'q'), 31) -> 31 
                                             
  Qubit(QuantumRegister(133, 'q'), 32) -> 32 
                                             
  Qubit(QuantumRegister(133, 'q'), 33) -> 33 
                                             
  Qubit(QuantumRegister(133, 'q'), 34) -> 34 
                                             
  Qubit(QuantumRegister(133, 'q'), 35) -> 35 
                                             
  Qubit(QuantumRegister(133, 'q'), 36) -> 36 
                                             
  Qubit(QuantumRegister(133, 'q'), 37) -> 37 
                                             
  Qubit(QuantumRegister(133, 'q'), 38) -> 38 
                                             
  Qubit(QuantumRegister(133, 'q'), 39) -> 39 
                                             
  Qubit(QuantumRegister(133, 'q'), 40) -> 40 
                                             
  Qubit(QuantumRegister(133, 'q'), 41) -> 41 
                                             
  Qubit(QuantumRegister(133, 'q'), 42) -> 42 
                                             
  Qubit(QuantumRegister(133, 'q'), 43) -> 43 
  